In [31]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [32]:
nav = pd.read_csv("data/raw/02_nav_history.csv")

transactions = pd.read_csv(
    "data/raw/08_investor_transactions.csv"
)

performance = pd.read_csv(
    "data/raw/07_scheme_performance.csv"
)

fund_master = pd.read_csv(
    "data/raw/01_fund_master.csv"
)

In [33]:
print(nav.shape)
print(nav.head())

print(nav.info())

(46000, 3)
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692
<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  str    
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 1.1 MB
None


In [34]:
nav["date"] = pd.to_datetime(
    nav["date"],
    errors="coerce"
)

In [35]:
nav["date"].dtype

dtype('<M8[us]')

In [36]:
nav = nav.sort_values(
    ["amfi_code","date"]
)

In [37]:
nav["nav"] = (
    nav
    .groupby("amfi_code")["nav"]
    .ffill()
)

In [38]:
nav = nav.drop_duplicates()

In [39]:
nav = nav.drop_duplicates(
    subset=["amfi_code","date"]
)

In [40]:
invalid_nav = nav[nav["nav"]<=0]

In [41]:
nav = nav[nav["nav"]>0]

In [42]:
nav.to_csv(
    "data/processed/nav_history.csv",
    index=False
)

In [43]:
transactions["transaction_type"] = (
transactions["transaction_type"]
.str.strip()
.str.lower()
)

In [44]:
mapping = {

"sip":"SIP",

"systematic investment":"SIP",

"lumpsum":"Lumpsum",

"lump sum":"Lumpsum",

"redemption":"Redemption"

}

In [45]:
transactions["transaction_type"] = (
transactions["transaction_type"]
.replace(mapping)
)

In [46]:
transactions = transactions[
transactions["amount_inr"]>0
]

In [47]:
transactions["transaction_date"] = pd.to_datetime(

transactions["transaction_date"],

errors="coerce"
)

In [48]:
valid = [
"Verified",
"Pending",
"Rejected"
]

invalid = transactions[
~transactions["kyc_status"].isin(valid)
]

In [49]:
transactions.to_csv(
"data/processed/investor_transactions.csv",
index=False
)

In [50]:
cols = [
"return_1yr_pct",
"return_3yr_pct",
"return_5yr_pct"
]

for c in cols:

    performance[c]=pd.to_numeric(
        performance[c],
        errors="coerce"
    )

In [51]:
performance[
performance["return_1yr_pct"]>200
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [52]:
performance[
performance["return_1yr_pct"]<-100
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [53]:
performance[
(performance["expense_ratio_pct"]<0.1)
|
(performance["expense_ratio_pct"]>2.5)
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [54]:
performance.to_csv(
"data/processed/scheme_performance.csv",
index=False
)

In [57]:
import sqlite3

conn = sqlite3.connect("database/bluestock_mf.db")
print("Database created successfully!")

conn.close()

Database created successfully!


In [60]:
import sqlite3

conn = sqlite3.connect("database/bluestock_mf.db")

with open("sql/schema.sql", "r") as f:
    conn.executescript(f.read())

print("Tables created successfully!")

conn.close()

Tables created successfully!


In [61]:
import sqlite3

conn = sqlite3.connect("database/bluestock_mf.db")

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

print(cursor.fetchall())

conn.close()

[]
